In [5]:
from langgraph.checkpoint.memory import MemorySaver
import operator
import os


from dotenv import load_dotenv
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from langchain.tools import tool

load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()
class SharedState(TypedDict):
    topic:    str
    messages: Annotated[List, add_messages]

class ReasearchState(SharedState):
    pass


class SummaryState(TypedDict):
    messages: Annotated[List, add_messages]

class ReasearchResponse(BaseModel):
    messages: List

system_prompt= "As expert professional, u read the topic name and shared 3 facts."
summary_system_prompt= "As expert professional reads all messages, writes a one-sentence summary"

def fetch_node(state:ReasearchState):
    response = llm_openai.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=state["topic"])
    ])
    return {"messages": [response]}

res_graph=StateGraph(ReasearchState)
res_graph.add_node(fetch_node)
res_graph.add_edge(START,'fetch_node')
res_graph.add_edge('fetch_node',END)

research_app = res_graph.compile()

def summary_node(state:SharedState):
    message=state["messages"]
    response=llm_openai.invoke([SystemMessage(content=summary_system_prompt)]+ state["messages"]).content
    return {"messages":response}

combine_graph=StateGraph(SharedState)
combine_graph.add_node('research_app',research_app)
combine_graph.add_node('summary_node',summary_node)
combine_graph.add_edge(START,'research_app')
combine_graph.add_edge('research_app','summary_node')
combine_graph.add_edge('summary_node',END)

app=combine_graph.compile()
result = app.invoke({"topic": "quantum computing", "messages": []})
print(f"Messages count: {len(result['messages'])}")
for msg in result['messages']:
    print(f"  {type(msg).__name__}: {msg.content[:80]}")




Messages count: 2
  AIMessage: **Three key facts about quantum computing**

1. **Qubits and Superposition** – U
  HumanMessage: Quantum computing exploits qubits’ superposition and entanglement to tackle cert


In [9]:
from langgraph.checkpoint.memory import MemorySaver
import operator
import os


from dotenv import load_dotenv
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from langchain.tools import tool

load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()
class SupervisorState(TypedDict):
    topic:    str
    messages: Annotated[List, add_messages]
    next:     str

class RouteDecision(BaseModel):
    next: Literal["research_agent", "writer_agent", "FINISH"]

system_prompt = """You are a supervisor managing a research and writing task.
Follow this exact workflow based on message count:
- If there are 0 messages: route to research_agent (research not done yet)
- If there is 1 message: route to writer_agent (research done, writing not done)
- If there are 2 or more messages: route to FINISH (both done)
"""

def supervisor_node(state:SupervisorState):
    msg_count = len(state["messages"])
    struc_res = llm_openai.with_structured_output(RouteDecision)
    response = struc_res.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Topic: {state['topic']}. Messages so far: {msg_count}")
    ])
    return {"next": response.next}

class ReasearchState(SupervisorState):
    pass

res_system_prompt= "As expert professional, u read the topic name and shared 3 facts."
def research_node(state:ReasearchState):
    response = llm_openai.invoke([
        SystemMessage(content=res_system_prompt),
        HumanMessage(content=state["topic"])
    ])
    return {"messages": [response]}


res_graph=StateGraph(ReasearchState)
res_graph.add_node('research_node',research_node)
res_graph.add_edge(START,'research_node')
res_graph.add_edge('research_node',END)

research_app = res_graph.compile()

class WriterState(SupervisorState):
    pass

writer_system_prompt= "As expert professional reads all messages, writes a short article on it"
def writer_node(state:WriterState):
    response=llm_openai.invoke([SystemMessage(content=writer_system_prompt)]+ state["messages"])
    return {"messages":[response]}

def condition(state:SupervisorState):
    message=state["next"]
    if message=="research_agent":
        return "research_app"
    elif message=="writer_agent":
        return "wri_app"
    else:
        return "FINISH"


wri_graph=StateGraph(WriterState)
wri_graph.add_node('writer_node',writer_node)
wri_graph.add_edge(START,'writer_node')
wri_graph.add_edge('writer_node',END)

wri_app = wri_graph.compile()



combine_graph=StateGraph(SupervisorState)
combine_graph.add_node('supervisor_node',supervisor_node)
combine_graph.add_node('research_app',research_app)
combine_graph.add_node('wri_app',wri_app)
combine_graph.add_edge(START,'supervisor_node')
combine_graph.add_conditional_edges('supervisor_node',condition,{"research_app":"research_app","wri_app":"wri_app","FINISH":END})
combine_graph.add_edge('research_app','supervisor_node')
combine_graph.add_edge("wri_app", "supervisor_node")

app=combine_graph.compile()
result = app.invoke({"topic": "black holes", "messages": [], "next": ""})
print(f"Total messages: {len(result['messages'])}")
for i, msg in enumerate(result['messages']):
    print(f"[{i+1}] {msg.content[:100]}")





C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouteDecision(next='research_agent'), input_type=RouteDecision])
  return self.__pydantic_serializer__.to_python(
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouteDecision(next='writer_agent'), input_type=RouteDecision])
  return self.__pydantic_serializer__.to_python(


Total messages: 2
[1] **Three key facts about black holes**

1. **Event Horizon as a Point of No Return**  
   The event h
[2] ## How an Expert Professional Turns a Stream of Messages into a Concise Article  

In today’s fast‑p


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RouteDecision(next='FINISH'), input_type=RouteDecision])
  return self.__pydantic_serializer__.to_python(


In [4]:
from langgraph.checkpoint.memory import MemorySaver
import operator
import os


from dotenv import load_dotenv
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from langchain.tools import tool
from langgraph.types import Command

load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()
class HandoffState(TypedDict):
    topic:    str
    messages: Annotated[List, add_messages]

class ReasearchState(HandoffState):
    pass

res_system_prompt= "As expert professional, u read the topic name and shared 3 facts."
def research_node(state:ReasearchState):
    response = llm_openai.invoke([
        SystemMessage(content=res_system_prompt),
        HumanMessage(content=state["topic"])
    ])
    return Command(
    update={"messages": [response]},
    goto="writer_node"
)



class WriterState(HandoffState):
    pass

writer_system_prompt= "As expert professional reads all messages, writes a short article on it"
def writer_node(state:WriterState):
    response=llm_openai.invoke([SystemMessage(content=writer_system_prompt)]+ state["messages"])
    return Command(
    update={"messages": [response]},
    goto=END
)


res_graph=StateGraph(HandoffState)
res_graph.add_node('research_node',research_node)
res_graph.add_node('writer_node',writer_node)
res_graph.add_edge(START,'research_node')

app = res_graph.compile()
result = app.invoke({"topic": "neural networks", "messages": []})
print(f"Messages: {len(result['messages'])}")
print(result['messages'][-1].content[:200])




Messages: 2
**Neural Networks: A Concise Expert Overview**

*By [Your Name], Ph.D., Machine‑Learning Researcher*  

---

### Introduction  

Artificial neural networks (ANNs) have become the cornerstone of modern
